In [1]:
### Use this cell because without it notebook crashed with tmp files
import os
import torch._dynamo

TASK_ROOT = "/mnt/newdata/dpanc/benchmarking/GENA_LM"

os.environ["TMPDIR"] = f"{TASK_ROOT}/cache/tmp"
os.environ["TRITON_CACHE_DIR"] = f"{TASK_ROOT}/cache/triton"
os.environ["TORCHINDUCTOR_CACHE_DIR"] = f"{TASK_ROOT}/cache/torchinductor"

for p in [
    os.environ["TMPDIR"],
    os.environ["TRITON_CACHE_DIR"],
    os.environ["TORCHINDUCTOR_CACHE_DIR"],
]:
    os.makedirs(p, exist_ok=True)

torch._dynamo.config.suppress_errors = True

In [2]:
# user-configurable variables

GENA_HOME = "/home/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch"
EXPERIMENT_CONFIG = "/mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/downstream_tasks/expression_prediction/inference_example/inference.yaml"
CHECKPOINT_PATH = "/mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM/models/full_model/pytorch_model.bin"

INFERENCE_DIR = None  # if None, use <GENA_HOME>/downstream_tasks/expression_prediction/inference_example
JSON_DIR = "/mnt/newdata/dpanc/benchmarking/GENA_LM/notebook_inference_valid812/json_14"
FORWARD_INTERVALS_PATH = "/mnt/newdata/dpanc/benchmarking/data/human.valid.forward.csv"
REVERSE_INTERVALS_PATH = "/mnt/newdata/dpanc/benchmarking/data/human.valid.reverse.csv"

GENOME_PATH = "/mnt/newdata/dpanc/benchmarking/data/hg38.fna"
NUM_BEFORE = 250
TOKEN_LEN_FOR_FETCH = 15

DNA_TOKENIZER = None  # if None, use gen_tokenizer from config
TEXT_TOKENIZER = None  # if None, use text_tokenizer from config
DNA_MAX_SEQ_LEN = None  # if None, use input_seq_len from config
TEXT_MAX_SEQ_LEN = None  # if None, use text_max_seq_len from config

PREDICTION_MATRIX_CSV = "predicted_expression_matrix.csv"
CORR_METHOD = "pearson"


In [3]:
import sys, os
import torch
import json
from pathlib import Path
from transformers import AutoTokenizer

# hydra imports; not really required if you will hard-code model params in future
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

# set GENALM_HOME environment variable to point to GENA_LM repo root; required to process config files
os.environ["GENALM_HOME"] = GENA_HOME 

sys.path.append(GENA_HOME)
sys.path.append(GENA_HOME+"/GENA_LM")

# import model
from downstream_tasks.expression_prediction.expression_model_final import ExpressionCounts
from downstream_tasks.expression_prediction.expression_dataset_final import ExpressionDataset
from downstream_tasks.expression_prediction.inference_example.inference_input_utils import prepare_inference_inputs_from_intervals

/home/dpanc/benchmarking/GENA_LM/envs/expression_flash/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# we have model parameters and other variables in config files; I made one for inference
experiment_config = EXPERIMENT_CONFIG

experiment_config_path = Path(experiment_config).expanduser().absolute()

with initialize_config_dir(str(experiment_config_path.parents[0])):
	experiment_config = compose(config_name=experiment_config_path.name)

model_kwargs = instantiate(experiment_config["model_kwargs"])

# initialize model
model = ExpressionCounts(**model_kwargs)

/tmp/ipykernel_3113920/842616341.py:6: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize_config_dir(str(experiment_config_path.parents[0])):
Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `torch_dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", torch_dtype=torch.float16)`


Using ModernGENA from /home/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/models/modernbert_large/
missing: 0 []
unexpected: 3 ['decoder.bias', 'head.dense.weight', 'head.norm.weight']
mismatched: []
bert dropouts: {'attention_dropout': 0.1, 'embedding_dropout': 0.1, 'mlp_dropout': 0.1}
qwen dropouts: {'attention_dropout': 0.1}
[desc_model] unfrozen transformer blocks: [24, 25, 26, 27] (total blocks=28)
[desc_model] backbone.norm trainable: True (trainable params=1,024)
[desc_model] trainable params: 62,924,800 / 595,776,512
[desc_model] trainable tensors: 45
  - layers.24.self_attn.q_proj.weight
  - layers.24.self_attn.k_proj.weight
  - layers.24.self_attn.v_proj.weight
  - layers.24.self_attn.o_proj.weight
  - layers.24.self_attn.q_norm.weight
  - layers.24.self_attn.k_norm.weight
  - layers.24.mlp.gate_proj.weight
  - layers.24.mlp.up_proj.weight
  - layers.24.mlp.down_proj.weight
  - layers.24.input_layernorm.weight
  - layers.24.post_attention_layernorm.weight
  - layers.25

In [5]:
# load checkpoint
checkpoint_path = CHECKPOINT_PATH 
model.load_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=True))

<All keys matched successfully>

In [6]:
# configure path-based inference inputs
# descriptions are loaded from all JSON files in json_dir,
# genes are taken from forward intervals (and optionally reverse intervals)

inference_dir = Path(INFERENCE_DIR) if INFERENCE_DIR is not None else Path(GENA_HOME) / "GENA_LM/downstream_tasks/expression_prediction/inference_example"
data_dir = inference_dir / "data"
json_dir = Path(JSON_DIR) if Path(JSON_DIR).is_absolute() else inference_dir / JSON_DIR
forward_intervals_path = Path(FORWARD_INTERVALS_PATH) if Path(FORWARD_INTERVALS_PATH).is_absolute() else inference_dir / FORWARD_INTERVALS_PATH
reverse_intervals_path = None if REVERSE_INTERVALS_PATH is None else (Path(REVERSE_INTERVALS_PATH) if Path(REVERSE_INTERVALS_PATH).is_absolute() else inference_dir / REVERSE_INTERVALS_PATH)
genome_path = Path(GENOME_PATH) 
num_before = int(NUM_BEFORE) 
token_len_for_fetch = TOKEN_LEN_FOR_FETCH


In [7]:
# helper that prepares descriptions from a folder with JSON files
# and tokenizes interval files exactly with the dataset logic

def prepare_inference_inputs(
	json_dir,
	forward_intervals_path,
	genome_path,
	reverse_intervals_path=None,
	gen_tokenizer=None,
	text_tokenizer_override=None,
	gen_max_seq_len_override=None,
	text_max_seq_len_override=None,
):
	gen_tokenizer_used = gen_tokenizer or dna_tokenizer
	text_tokenizer_used = text_tokenizer_override or text_tokenizer
	gen_max_seq_len_used = gen_max_seq_len_override or dna_max_seq_len
	text_max_seq_len_used = text_max_seq_len_override or text_max_seq_len

	return prepare_inference_inputs_from_intervals(
		json_dir=json_dir,
		forward_intervals_path=forward_intervals_path,
		reverse_intervals_path=reverse_intervals_path,
		genome_path=genome_path,
		gen_tokenizer=gen_tokenizer_used,
		text_tokenizer=text_tokenizer_used,
		gen_max_seq_len=gen_max_seq_len_used,
		text_max_seq_len=text_max_seq_len_used,
		cache_dir=inference_dir,
		num_before=num_before,
		token_len_for_fetch=token_len_for_fetch,
	)


In [8]:
# prepare tokenizers
dna_tokenizer_name = DNA_TOKENIZER or experiment_config["args_params"]["gen_tokenizer"]
text_tokenizer_name = TEXT_TOKENIZER or experiment_config["shared_dataset_params"]["text_tokenizer"]
dna_tokenizer = AutoTokenizer.from_pretrained(dna_tokenizer_name)
text_tokenizer = AutoTokenizer.from_pretrained(text_tokenizer_name, padding_side='left')

dna_max_seq_len = int(DNA_MAX_SEQ_LEN) if DNA_MAX_SEQ_LEN is not None else int(experiment_config["args_params"]["input_seq_len"])
text_max_seq_len = int(TEXT_MAX_SEQ_LEN) if TEXT_MAX_SEQ_LEN is not None else int(experiment_config["shared_dataset_params"]["text_max_seq_len"])

In [9]:
# prepare interval-based inference inputs

prepared_inference = prepare_inference_inputs(
	json_dir=json_dir,
	forward_intervals_path=forward_intervals_path,
	genome_path=genome_path,
	reverse_intervals_path=reverse_intervals_path,
)

genes = prepared_inference["genes"]
experiments = prepared_inference["experiments"]
tokenized_DNA = prepared_inference["tokenized_DNA"]
tokenized_descriptions = prepared_inference["tokenized_descriptions"]

print("Gene token caches:", prepared_inference["gene_cache_paths"])
print("Description token cache:", prepared_inference["description_cache_path"])
print("Input IDs shape:", tokenized_DNA["input_ids"].shape)
tokenized_DNA


Gene token caches: {'forward': '/mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/GENA_LM/downstream_tasks/expression_prediction/inference_example/inference_dataset_hash.forward.4f37f4754b2797cc.h5', 'reverse': '/mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/GENA_LM/downstream_tasks/expression_prediction/inference_example/inference_dataset_hash.reverse.b4ec97b0cf922465.h5'}
Description token cache: /mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/GENA_LM/downstream_tasks/expression_prediction/inference_example/json_14.9f8350900d93d327.Qwen_Qwen3-Embedding-0.6B.510.description.h5
Input IDs shape: torch.Size([3038, 1024])


{'input_ids': tensor([[    1,   376,  1569,  ...,     3,     3,     3],
         [    1,   264,   511,  ...,   561, 29117,     2],
         [    1,    51,   527,  ...,   222,  3550,     2],
         ...,
         [    1,   110,   572,  ...,   377,   386,     2],
         [    1,   922,  1351,  ...,    71,   231,     2],
         [    1,  4117,  2213,  ...,  2858,  3722,     2]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]),
 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]])}

In [10]:
# inspect tokenized experiment description

first_experiment = next(iter(experiments))
print("Experiments:", list(experiments.keys()))
print("Input IDs shape:", tokenized_descriptions[first_experiment]["input_ids"].shape)
tokenized_descriptions[first_experiment]


Experiments: ['ENCFF035CWS', 'ENCFF083EOC', 'ENCFF123KIW', 'ENCFF236XOK', 'ENCFF242BWW', 'ENCFF329ENM', 'ENCFF361XCF', 'ENCFF494KRC', 'ENCFF602HCV', 'ENCFF660EXG', 'ENCFF664WLU', 'ENCFF761SPP', 'ENCFF784MDF', 'ENCFF857JQM']
Input IDs shape: torch.Size([3038, 135])


{'input_ids': tensor([[   395,    352,   4647,  ...,     23,     13, 151643],
         [   395,    352,   4647,  ...,     23,     13, 151643],
         [   395,    352,   4647,  ...,     23,     13, 151643],
         ...,
         [   395,    352,   4647,  ...,     23,     13, 151643],
         [   395,    352,   4647,  ...,     23,     13, 151643],
         [   395,    352,   4647,  ...,     23,     13, 151643]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]])}

In [ ]:
# cast inputs and run forward pass
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# in this example B=len(genes), N=1, L=len(DNA tokens), D=len(text tokens)

input_ids = tokenized_DNA["input_ids"].to(device) # (B*N, L)
attention_mask = tokenized_DNA["attention_mask"].to(device) # (B*N, L)

# we use dummy labels and labels_mask as we don't need them for inference
labels_mask = torch.ones_like(input_ids, device=device, dtype=torch.bool).unsqueeze(-1).to(device) # (B*N, L, 1)
labels = torch.ones_like(input_ids, device=device, dtype=torch.float32).unsqueeze(-1).to(device) # (B*N, L, 1)


model = model.eval()
model.to(device)

outputs = {}

for experiment in experiments:
	# we add extra dim for N=1
	desc_input_ids = torch.unsqueeze(tokenized_descriptions[experiment]["input_ids"], dim=1).to(device)  # (B, N, D)

	desc_attention_mask = torch.unsqueeze(tokenized_descriptions[experiment]["attention_mask"], dim=1).to(device) # (B, N, D)

	dataset_flag = torch.zeros(size=(input_ids.shape[0], 1), device=device, dtype=torch.bool) # (B, N): 1 -> repeating DNA; 0 -> repeating DESC; we repeat DESC here


	with torch.autocast(device_type="cuda", dtype=torch.bfloat16), torch.no_grad():
			output = model(input_ids=input_ids, attention_mask=attention_mask, desc_input_ids=desc_input_ids, desc_attention_mask=desc_attention_mask, dataset_flag=dataset_flag)

	outputs[experiment] = output

# def forward(
# 	self,
# 	input_ids=None,              # (B*N, L)
# 	attention_mask=None,         # (B*N, L) or None
# 	labels_mask=None,            # (B*N, L, 1)
# 	labels=None,                 # (B*N, L, 1)
# 	return_dict=None,
# 	desc_input_ids=None,           # (B, N, D)
# 	desc_attention_mask = None,
# 	dataset_flag=None,           # (B, N): 1 -> дубли INPUTS; 0 -> дубли DESC
# ):

RuntimeError: CUDA error: an illegal memory access was encountered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


: 

In [16]:
# build prediction table and gene x cell-type matrix

import pandas as pd

prediction_rows = []
gene_names = list(genes.keys())
cell_type_names = list(experiments.keys())

for cell_type_name in cell_type_names:
	for gene_idx, gene_name in enumerate(gene_names):
		predicted_expression = outputs[cell_type_name]["logits"][gene_idx, 0, 0].item()
		prediction_rows.append(
			{
				"Cell Type": cell_type_name,
				"Gene": gene_name,
				"Predicted Expression": predicted_expression,
			}
		)

predictions_df = pd.DataFrame(prediction_rows)
expression_matrix = predictions_df.pivot(index="Gene", columns="Cell Type", values="Predicted Expression")
expression_matrix = expression_matrix.reindex(index=gene_names, columns=cell_type_names)

display(predictions_df)
display(expression_matrix)


,Cell Type,Gene,Predicted Expression
0,ENCFF081FQX,AL669831.5,0.478516
1,ENCFF081FQX,FAM87B,0.011353
2,ENCFF081FQX,AL645608.6,0.091309
3,ENCFF081FQX,AL645608.2,0.045898
4,ENCFF081FQX,AL645608.4,0.047852
5,ENCFF578UUD,AL669831.5,2.218750
6,ENCFF578UUD,FAM87B,0.051758
7,ENCFF578UUD,AL645608.6,0.106934
8,ENCFF578UUD,AL645608.2,0.032227
9,ENCFF578UUD,AL645608.4,0.095215


Cell Type,ENCFF081FQX,ENCFF578UUD,ENCFF588KDY
Gene,,,
AL669831.5,0.478516,2.218750,1.218750
FAM87B,0.011353,0.051758,0.022583
AL645608.6,0.091309,0.106934,0.095215
AL645608.2,0.045898,0.032227,0.074219
AL645608.4,0.047852,0.095215,0.189453


In [14]:
# save gene x cell-type table to CSV

prediction_matrix_csv = Path(PREDICTION_MATRIX_CSV)
if not prediction_matrix_csv.is_absolute():
	prediction_matrix_csv = inference_dir / prediction_matrix_csv

expression_matrix.to_csv(prediction_matrix_csv)
print(f"Saved prediction matrix to: {prediction_matrix_csv}")


Saved prediction matrix to: /home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/inference_example/predicted_expression_matrix.csv
